# Entrate Stato 2008-2024 - composizione per Titolo

Prima lettura sulla composizione delle entrate dello Stato con focus su 2008, 2020, 2024.

**Fonte**: MEF / RGS OpenBDAP — Rendiconto pubblicato entrate Stato  
**Dati**: [dataciviclab-clean](https://storage.googleapis.com/dataciviclab-clean/bdap_entrate_stato/2024/bdap_entrate_stato_2024_clean.parquet) (pubblico, CC BY)  
**Caveat**: le misure sono Previsioni Definitive CP/CS da rendiconto, non incasso effettivo.

In [ ]:
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False

GCS_URL = 'https://storage.googleapis.com/dataciviclab-clean/bdap_entrate_stato/2024/bdap_entrate_stato_2024_clean.parquet'
ANNI_FOCUS = [2008, 2020, 2024]

df = pd.read_parquet(GCS_URL)
df = df.rename(columns={'esercizio_finanziario': 'anno'})
df.head()

In [ ]:
# Titolo breve: prende la parte dopo il trattino, es. "TITOLO I - ENTRATE TRIBUTARIE" -> "ENTRATE TRIBUTARIE"
df['titolo_breve'] = df['titolo'].str.split(' - ', n=1).str[-1].str.strip()

# Aggregazione per anno e titolo
agg = (
    df[df['anno'].isin(ANNI_FOCUS)]
    .groupby(['anno', 'titolo_breve'], as_index=False)['previsioni_definitive_cp']
    .sum()
)

# Quota sul totale annuo
totale_anno = agg.groupby('anno')['previsioni_definitive_cp'].transform('sum')
agg['quota_cp'] = agg['previsioni_definitive_cp'] / totale_anno
agg = agg.sort_values(['anno', 'titolo_breve'])
agg

In [ ]:
pivot = agg.pivot(index='anno', columns='titolo_breve', values='quota_cp').sort_index()
if HAS_MATPLOTLIB:
    ax = pivot.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab20c')
    ax.set_title('Quota CP per Titolo - anni focus')
    ax.set_xlabel('Anno')
    ax.set_ylabel('Quota sul totale annuo')
    ax.legend(title='Titolo', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    pivot

In [ ]:
confronto = agg[agg['titolo_breve'].isin(['ENTRATE TRIBUTARIE', 'ACCENSIONE DI PRESTITI'])].copy()
confronto = confronto.sort_values(['anno', 'titolo_breve'])
confronto

## Prima lettura

Sulla quota CP aggregata per i due titoli principali:

| Anno | Entrate tributarie | Accensione prestiti |
|------|--------------------|--------------------|
| 2008 | ~62%               | ~33%               |
| 2020 | ~43%               | ~50%               |
| 2024 | ~49%               | ~42%               |

Nelle due crisi principali del periodo (2008-2009 e 2020) la composizione si sposta in modo marcato verso l'accensione di prestiti. Nel 2024 si osserva un parziale rientro, con le entrate tributarie che recuperano quota.

**Caveat principale**: le misure sono Previsioni Definitive da rendiconto (CP = Competenza Propria), non incassi effettivi. La lettura riguarda il perimetro autorizzato a consuntivo, non la riscossione reale.

**Cosa non possiamo ancora dire**:
- se la variazione dipenda da riclassificazioni contabili in alcuni anni
- se il fenomeno sia una scelta strutturale o una risposta una tantum alle crisi
- effetti su finanza pubblica allargata (perimetro solo Stato centrale)